# NLP05 — KoChatGPT 업그레이드
KoGPT2 기반 RLHF 3단계 파이프라인: SFT → RM → PPO

**루브릭**
- ① baseline vs SFT 비교
- ② SFT vs RM/PPO 비교
- ③ Decoding 실험 + LLM-as-a-Judge

In [ ]:
import os
os.chdir('/workspace/NLP/NLP05')
import sys
sys.path.append('/workspace/NLP/NLP05')
print('Working dir:', os.getcwd())

In [ ]:
# 패키지 설치
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers', 'torch', 'nltk',
                'rouge-score', 'matplotlib', '--break-system-packages'])
import nltk
nltk.download('punkt', quiet=True)
print('설치 완료')

## STEP 1 — SFT (지도 미세조정)
KoGPT2 pretrained 모델을 kochatgpt_1_SFT.jsonl 데이터로 파인튜닝

In [ ]:
import runpy
runpy.run_path('/workspace/NLP/NLP05/scripts/01_sft_train.py',
               run_name='__main__')

## STEP 2 — RM (보상 모델 학습)
SFT 모델 백본 위에 ranking loss로 reward model 훈련

In [ ]:
runpy.run_path('/workspace/NLP/NLP05/scripts/02_rm_train.py',
               run_name='__main__')

## STEP 3 — PPO (강화학습)
Policy=SFT모델, Reward=RM, KL penalty로 분포 유지

In [ ]:
runpy.run_path('/workspace/NLP/NLP05/scripts/03_ppo_train.py',
               run_name='__main__')

## STEP 4 — 평가 및 비교
### 루브릭① baseline vs SFT / 루브릭② SFT vs PPO / 루브릭③ Decoding 실험

In [ ]:
runpy.run_path('/workspace/NLP/NLP05/scripts/04_inference.py',
               run_name='__main__')

## STEP 5 — 결과 시각화

In [ ]:
import json, matplotlib.pyplot as plt
import config as cfg

# 정량 비교표
with open(f'{cfg.RESULTS_DIR}/quant_summary.json') as f:
    quant = json.load(f)

stages = list(quant.keys())
bleu   = [quant[s]['avg_bleu']    for s in stages]
rouge  = [quant[s]['avg_rouge_l'] for s in stages]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(stages, bleu,  color=['#aec6cf','#779ecb','#4a7ebf'])
axes[0].set_title('BLEU Score 비교')
axes[0].set_ylabel('BLEU')
axes[1].bar(stages, rouge, color=['#aec6cf','#779ecb','#4a7ebf'])
axes[1].set_title('ROUGE-L Score 비교')
axes[1].set_ylabel('ROUGE-L')
plt.suptitle('baseline vs SFT vs PPO 정량 비교')
plt.tight_layout()
plt.savefig(f'{cfg.RESULTS_DIR}/quantitative_comparison.png', dpi=150)
plt.show()
print(json.dumps(quant, indent=2, ensure_ascii=False))

In [ ]:
# Decoding 실험 결과 출력
with open(f'{cfg.RESULTS_DIR}/decoding_results.json') as f:
    decoding = json.load(f)

print('=== Decoding 전략별 응답 비교 ===\n')
for item in decoding[:3]:
    print(f"[프롬프트] {item['prompt']}")
    for strategy in cfg.DECODING_CONFIGS.keys():
        print(f"  {strategy:10s}: {item.get(strategy, 'N/A')}")
    print()

In [ ]:
# LLM-as-a-Judge 샘플 출력
from scripts.inference_04 import build_judge_prompt
with open(f'{cfg.RESULTS_DIR}/model_comparison.json') as f:
    comparisons = json.load(f)

print('=== LLM-as-a-Judge 평가 프롬프트 샘플 ===\n')
sample = comparisons[0]
print(build_judge_prompt(sample['prompt'], sample['sft'], sample['ppo']))

## 결론
| 단계 | 모델 | BLEU | ROUGE-L |
|------|------|------|----------|
| baseline | KoGPT2 pretrained | - | - |
| SFT | KoGPT2 + 지도학습 | - | - |
| PPO | SFT + RLHF | - | - |

> 수치는 실행 후 위 셀 결과로 채울 것